# Registry-driven packing experiment
The same notebook runs any implemented level/algorithm locally, on Colab, or on Kaggle. It contains no model implementation.

In [1]:
from pathlib import Path
import sys
start = Path.cwd().resolve()
root = next((p for p in (start, *start.parents) if (p / 'pyproject.toml').exists()), None)
if root is None:
    raise RuntimeError('Clone/open the repository before running this notebook')
sys.path.insert(0, str(root / 'src'))
root

WindowsPath('D:/IT/Project/gsm/3DBinPacking')

In [2]:
from container_packing.algorithms.registry import list_algorithms
from container_packing.levels.registry import list_levels
from container_packing.runtime.inputs import prompt_choice, prompt_positive

LEVEL_ID = 'level_01'
ALGORITHM_ID = 'milp_big_m'
ITEM_COUNT = 20
CONTAINER_COUNT = 5
ENVIRONMENT = 'local'  # local, colab, or kaggle
INTERACTIVE_INPUT = False

if INTERACTIVE_INPUT:
    level_ids = tuple(x.level_id for x in list_levels())
    LEVEL_ID = prompt_choice('Level', level_ids, LEVEL_ID)
    algorithm_ids = tuple(x.algorithm_id for x in list_algorithms(level_id=LEVEL_ID))
    ALGORITHM_ID = prompt_choice('Algorithm', algorithm_ids, ALGORITHM_ID)
    ITEM_COUNT = prompt_positive('Number of items', ITEM_COUNT)
    CONTAINER_COUNT = prompt_positive('Number of containers', CONTAINER_COUNT)
    ENVIRONMENT = prompt_choice('Environment', ('local', 'colab', 'kaggle'), ENVIRONMENT)
print(LEVEL_ID, ALGORITHM_ID, ITEM_COUNT, CONTAINER_COUNT, ENVIRONMENT)

level_01 milp_big_m 20 5 local


In [3]:
from container_packing.experiments.contracts import ExperimentRequest
from container_packing.experiments.runner import run_experiment
from container_packing.levels.registry import get_level

level = get_level(LEVEL_ID)
request = ExperimentRequest(
    level_id=LEVEL_ID, algorithm_id=ALGORITHM_ID,
    config_path=root / level.default_config,
    item_count=ITEM_COUNT, container_count=CONTAINER_COUNT,
    environment=ENVIRONMENT,
)
result = run_experiment(request)
result.metadata

{'status': 'OPTIMAL',
 'solver': 'scipy.optimize.milp/HiGHS',
 'instance_id': 'level_01_20items_5containers',
 'run_id': '20260721T031631361076Z__level_01__milp_big_m__i20_c5__seed42',
 'run_dir': 'outputs/level_01/runs/20260721T031631361076Z__level_01__milp_big_m__i20_c5__seed42',
 'level_id': 'level_01',
 'algorithm_id': 'milp_big_m',
 'environment': 'local',
 'random_seed': 42,
 'solver_message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)',
 'objective_value': 10992.0,
 'n_items': 20,
 'n_containers': 5,
 'n_pairs': 190,
 'n_variables': 5865,
 'n_constraints': 18475,
 'constraint_nnz': 48505,
 'big_m': {'x': 6000.0, 'y': 2500.0, 'z': 2800.0},
 'objective_priority_constant': 4591.0,
 'level': 1,
 'rotation_enabled': False,
 'stability_enabled': False,
 'stackability_enabled': False,
 'items_data_status': 'public benchmark sample',
 'containers_data_status': 'synthetic_level1',
 'cost_note': 'Synthetic comparison score; not a real freight price.',
 'container_cou

In [4]:
assert result.metadata['n_items'] == ITEM_COUNT
assert result.metadata['n_containers'] == CONTAINER_COUNT
if result.validation is not None:
    assert result.validation.valid
print('Run directory:', result.metadata['run_dir'])

Run directory: outputs/level_01/runs/20260721T031631361076Z__level_01__milp_big_m__i20_c5__seed42
